# Human3R: Everyone Everywhere All at Once
## Multi-Environment Demo (Google Colab & Local)

[![arXiv](https://img.shields.io/badge/Arxiv-2510.06219-b31b1b.svg?logo=arXiv)](https://arxiv.org/abs/2510.06219) 
[![Home Page](https://img.shields.io/badge/Project-Website-C27185.svg)](https://fanegg.github.io/Human3R)

This notebook demonstrates Human3R for multi-human 3D reconstruction from monocular videos.

**Environment Support:**
- ☁️ **Google Colab**: Requires GPU runtime (`Runtime > Change runtime type > GPU`)
- 💻 **Local Ubuntu**: Requires NVIDIA GPU with CUDA support

**Updated:** Now using PyTorch 2.5.1 for improved compatibility.

## 1. Check GPU Availability

In [ ]:
import torch
import sys
import os

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
environment = "Google Colab" if IN_COLAB else "Local Machine"

print(f"🖥️  Environment: {environment}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"\n{'='*60}")

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA version: {torch.version.cuda}")
    capability = torch.cuda.get_device_capability(0)
    print(f"   GPU Compute Capability: {capability[0]}.{capability[1]}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: No GPU detected!")
    if IN_COLAB:
        print("\n📝 To enable GPU in Google Colab:")
        print("   1. Go to Runtime > Change runtime type")
        print("   2. Select 'T4 GPU' or 'A100 GPU' from Hardware accelerator")
        print("   3. Click Save and wait for runtime to restart")
    else:
        print("\n📝 For local Ubuntu machine:")
        print("   1. Verify NVIDIA GPU is installed: nvidia-smi")
        print("   2. Check CUDA installation: nvcc --version")
        print("   3. Install CUDA toolkit if needed:")
        print("      https://developer.nvidia.com/cuda-downloads")
        print("   4. Reinstall PyTorch with CUDA support:")
        print("      pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
        
        # Try to detect NVIDIA GPU
        try:
            import subprocess
            result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
            if result.returncode == 0:
                print("\n✓ NVIDIA GPU detected via nvidia-smi!")
                print("  Issue: PyTorch not compiled with CUDA support")
                print("  Solution: Reinstall PyTorch with CUDA (see command above)")
            else:
                print("\n✗ No NVIDIA GPU found (nvidia-smi failed)")
        except FileNotFoundError:
            print("\n✗ nvidia-smi not found - CUDA drivers may not be installed")

print(f"{'='*60}\n")

## 2. Mount Google Drive

In [ ]:
# Mount Google Drive (Colab only)
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted at /content/drive")
else:
    print("ℹ️  Skipping Google Drive mount (not in Colab environment)")
    print("   For local usage, use absolute paths to your video files")

## 3. Clone Repository and Install Dependencies

In [ ]:
# Clone the repository and checkout dev branch (Colab only)
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("☁️  Cloning repository for Colab environment...")
    !git clone https://github.com/atonalfreerider/Human3R.git
    %cd Human3R
    !git checkout dev
    print("✅ Repository cloned and switched to dev branch")
else:
    print("💻 Running on local machine - skipping repository clone")
    print("   Assuming we're already in the Human3R directory")
    print(f"   Current directory: {os.getcwd()}")
    
    # Verify we're in the right directory
    if not os.path.exists('src') or not os.path.exists('demo.py'):
        print("⚠️  Warning: Expected files not found. Make sure you're in the Human3R directory.")
    else:
        print("✅ Repository directory verified")

In [ ]:
# Install system dependencies (Colab only)
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("☁️  Installing system dependencies for Colab...")
    !apt-get update -qq
    !apt-get install -y -qq ffmpeg ninja-build
    print("✅ System dependencies installed")
else:
    print("💻 Running on local machine - skipping system package installation")
    print("   Ensure you have ffmpeg and ninja-build installed:")
    print("   sudo apt-get install ffmpeg ninja-build")

In [ ]:
# Install Python dependencies with PyTorch 2.5 (compatible version)
# PyTorch 2.8+ has compatibility issues with this model

# First uninstall any existing PyTorch packages
!pip uninstall -y torch torchvision torchaudio

# Install PyTorch 2.5.1 with matching torchaudio
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# Install other dependencies
!pip install -q "numpy>=1.26.4,<2.2" roma gradio matplotlib tqdm scipy einops trimesh tensorboard
!pip install -q "pyglet<2" "huggingface-hub[torch]>=0.22" viser lpips hydra-core "pillow>=10.3.0"
!pip install -q h5py accelerate transformers scikit-learn OpenEXR pyrender==0.1.45 smplx "pyvista>=0.44.0"
!pip install -q git+https://github.com/mattloper/chumpy@9b045ff5d6588a24a0bab52c83f032e2ba433e17
!pip install -q gdown

print("📦 Installing PyTorch 2.5.1 with matching torchaudio")
print("   Note: PyTorch 2.8+ has known compatibility issues with this model")
print("   Using CUDA 12.1 wheel from PyTorch official repository")

In [ ]:
# Fix opencv compatibility with numpy
# Reinstall opencv-python to be compatible with newer numpy versions
!pip uninstall -y opencv-python-headless opencv-contrib-python opencv-python
!pip install -q "opencv-python>=4.10.0"

# Verify installations
import torch
import numpy as np
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ NumPy version: {np.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

## 4. Apply PyTorch 2.8+ Compatibility Patch

This patch fixes model loading compatibility with PyTorch 2.8+.

In [ ]:
# Create PyTorch 2.8+ compatibility patches
import os

# Set CUDA environment variables for better error reporting and compatibility
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # Enable synchronous CUDA operations for debugging
os.environ['TORCH_USE_CUDA_DSA'] = '1'    # Enable device-side assertions

# Patch 1: torch.load compatibility
patch_torch_load = '''"""
PyTorch 2.8+ compatibility patch for torch.load
Adds weights_only=False to torch.load calls to support loading models with OmegaConf
"""

import torch
import functools

# Store the original torch.load function
_original_torch_load = torch.load

@functools.wraps(_original_torch_load)
def _patched_torch_load(*args, **kwargs):
    """
    Patched torch.load that adds weights_only=False for backward compatibility
    with PyTorch 2.8+ when loading models that contain custom objects like OmegaConf
    """
    # Only add weights_only if not already specified
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    
    try:
        return _original_torch_load(*args, **kwargs)
    except TypeError:
        # For older PyTorch versions that don't support weights_only parameter
        kwargs.pop('weights_only', None)
        return _original_torch_load(*args, **kwargs)

# Monkey-patch torch.load
torch.load = _patched_torch_load

print("✓ Applied PyTorch 2.8+ compatibility patch for model loading")
'''

os.makedirs('src', exist_ok=True)
with open('src/pytorch28_compat.py', 'w') as f:
    f.write(patch_torch_load)

print("✅ PyTorch 2.8+ compatibility patch created")
print("✅ CUDA debugging enabled (CUDA_LAUNCH_BLOCKING=1, TORCH_USE_CUDA_DSA=1)")

# Import the patches to apply them
import sys
sys.path.insert(0, 'src')
from pytorch28_compat import _patched_torch_load

print("✅ Compatibility patches applied and active")

## 5. Compile CUDA Kernels for RoPE

This step compiles custom CUDA kernels. PyTorch 2.5 has improved CUDA compilation support.

**Note:** If compilation fails, the model will work with PyTorch's built-in operations (minor performance impact).

In [ ]:
# Compile cuda kernels with proper architecture flags
import os
import torch
import subprocess
import sys

# Set CUDA architecture based on detected GPU
compilation_success = False

if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    cuda_arch = f"{capability[0]}.{capability[1]}"
    
    # Map common Colab GPUs to their architectures
    # T4: 7.5, P100: 6.0, V100: 7.0, A100: 8.0, L4: 8.9
    arch_list = f"{capability[0]}.{capability[1]}"
    os.environ['TORCH_CUDA_ARCH_LIST'] = arch_list
    
    print(f"Setting CUDA architecture: {arch_list}")
    print(f"GPU Compute Capability: {cuda_arch}")
    print(f"PyTorch version: {torch.__version__}")

    # Change to CUDA kernel directory
    original_dir = os.getcwd()
    os.chdir('src/croco/models/curope/')
    
    # Try to compile
    print("\nAttempting to compile CUDA kernels...")
    try:
        result = subprocess.run(
            [sys.executable, 'setup.py', 'build_ext', '--inplace'],
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if result.returncode == 0:
            compilation_success = True
            print("✅ CUDA kernels compiled successfully!")
        else:
            print("⚠️ CUDA kernel compilation failed.")
            print("The model will continue to work using PyTorch's built-in operations.")
            if "error:" in result.stderr:
                print("\nError details (first 500 chars):")
                print(result.stderr[:500])
    except subprocess.TimeoutExpired:
        print("⚠️ Compilation timed out after 5 minutes.")
    except Exception as e:
        print(f"⚠️ Unexpected error during compilation: {str(e)[:200]}")
    
    # Return to original directory
    os.chdir(original_dir)
else:
    print("⚠️ CUDA not available, skipping kernel compilation.")

if not compilation_success:
    print("\n" + "="*60)
    print("ℹ️  CUDA Kernel Compilation Status: SKIPPED")
    print("="*60)
    print("Don't worry! PyTorch has optimized built-in operations.")
    print("Performance impact is minimal for most use cases.")
    print("="*60)

In [ ]:
# Download Human3R checkpoint (copy from Drive in Colab, or download locally)
import os
import sys
import shutil

IN_COLAB = 'google.colab' in sys.modules
checkpoint_path = './src/human3r.pth'

if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
    print(f"✅ Human3R checkpoint already exists ({size_mb:.1f} MB)")
    print(f"   Location: {checkpoint_path}")
else:
    if IN_COLAB:
        # Copy from Google Drive
        drive_checkpoint = '/content/drive/MyDrive/SMPL/human3r.pth'
        
        if os.path.exists(drive_checkpoint):
            print(f"📥 Copying Human3R checkpoint from Google Drive...")
            shutil.copy2(drive_checkpoint, checkpoint_path)
            
            if os.path.exists(checkpoint_path):
                size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
                print(f"✅ Human3R checkpoint copied successfully ({size_mb:.1f} MB)")
            else:
                print("❌ Failed to copy checkpoint")
        else:
            print(f"❌ Checkpoint not found in Google Drive: {drive_checkpoint}")
            print("   Please ensure human3r.pth is in /content/drive/MyDrive/SMPL/")
    else:
        # Local machine: download from HuggingFace
        print("📥 Downloading Human3R checkpoint from HuggingFace...")
        !hf download faneggg/human3r human3r.pth --local-dir ./src
        
        if os.path.exists(checkpoint_path):
            size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
            print(f"✅ Human3R checkpoint downloaded successfully ({size_mb:.1f} MB)")
        else:
            print("❌ Failed to download checkpoint")

In [ ]:
# Setup SMPL models based on environment
import os
import sys
import shutil

IN_COLAB = 'google.colab' in sys.modules

os.makedirs('src/models/smplx', exist_ok=True)
os.makedirs('src/models/smpl', exist_ok=True)

# Define required files with their source paths in Google Drive
required_files = {
    'src/models/smplx/SMPLX_NEUTRAL.npz': ('body_models', 'SMPLX_NEUTRAL.npz', 'SMPL-X model'),
    'src/models/smpl/SMPL_NEUTRAL.pkl': ('smpl', 'SMPL_NEUTRAL.pkl', 'SMPL neutral model'),
    'src/models/smpl/SMPL_FEMALE.pkl': ('smpl', 'SMPL_FEMALE.pkl', 'SMPL female model'),
    'src/models/smpl/SMPL_MALE.pkl': ('smpl', 'SMPL_MALE.pkl', 'SMPL male model'),
    'src/models/smpl_mean_params.npz': ('multiHMR', 'smpl_mean_params.npz', 'SMPL mean parameters'),
    'src/models/smpl/J_regressor_h36m.npy': ('smpl', 'J_regressor_h36m.npy', 'Joint regressor'),
    'src/models/smplx/smplx2smpl.pkl': ('body_models', 'smplx2smpl.pkl', 'SMPL-X to SMPL converter')
}

if IN_COLAB:
    # Copy from Google Drive with new subdirectory structure
    base_dir = '/content/drive/MyDrive/SMPL/models'
    
    if not os.path.exists(base_dir):
        print(f"❌ Source directory not found: {base_dir}")
        print("   Please make sure SMPL models are in your Google Drive at:")
        print("   /content/drive/MyDrive/SMPL/models/")
        print("   With subdirectories: body_models/, smpl/, multiHMR/")
    else:
        print(f"📁 Copying SMPL models from: {base_dir}")
        
        # Copy all files from Google Drive subdirectories
        for dest_path, (subdir, filename, desc) in required_files.items():
            source_path = os.path.join(base_dir, subdir, filename)
            
            if os.path.exists(source_path):
                os.makedirs(os.path.dirname(dest_path), exist_ok=True)
                shutil.copy2(source_path, dest_path)
                size_kb = os.path.getsize(dest_path) / 1024
                print(f"  ✓ Copied {desc}: {filename} ({size_kb:.1f} KB)")
            else:
                print(f"  ✗ Not found: {source_path}")
else:
    # Local machine - check if files exist
    print("🔍 Checking for existing SMPL models...")
    
    missing_files = []
    for dest_path, (_, _, desc) in required_files.items():
        if os.path.exists(dest_path):
            size_kb = os.path.getsize(dest_path) / 1024
            print(f"  ✓ {desc}: {dest_path} ({size_kb:.1f} KB)")
        else:
            print(f"  ✗ {desc}: {dest_path} NOT FOUND")
            missing_files.append((dest_path, desc))
    
    if missing_files:
        print(f"\n⚠️  {len(missing_files)} file(s) missing")
        print("\nTo download missing files, you need to:")
        print("  1. Register at https://smpl-x.is.tue.mpg.de")
        print("  2. Register at https://smpl.is.tue.mpg.de")
        print("  3. Run the download script:")
        print("     cd /home/john/Desktop/3DPose/Human3R")
        print("     bash scripts/fetch_smplx.sh")
        print("\nOr manually download and place files in src/models/")
    else:
        print("\n✅ All SMPL models found!")

# Verify all files are present
print("\n🔍 Final verification:")
all_present = True
for dest_path, (_, _, desc) in required_files.items():
    if os.path.exists(dest_path):
        size_kb = os.path.getsize(dest_path) / 1024
        print(f"  ✓ {desc}: {os.path.basename(dest_path)} ({size_kb:.1f} KB)")
    else:
        print(f"  ✗ {desc}: {os.path.basename(dest_path)} MISSING")
        all_present = False

if all_present:
    print("\n✅ All SMPL models are ready!")
else:
    print("\n❌ Some SMPL models are missing - inference may fail")

In [ ]:
# Set video path based on environment
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Google Colab: Use Google Drive path
    video_path = '/content/drive/MyDrive/Zouk/3D-Pose/carlos-aline/carlos-aline-spin-cam1.mp4'
    print("☁️  Running in Google Colab")
else:
    # Local machine: Use local path
    video_path = '/home/john/Videos/Zouk/carlos-aline-spin-cam2.mp4'
    print("💻 Running on local machine")

print(f"📹 Video path: {video_path}")

# Verify the video file exists
if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    print(f"✅ Video found ({size_mb:.1f} MB)")
else:
    print(f"⚠️  Video not found!")
    print(f"   Expected location: {video_path}")
    if IN_COLAB:
        print("\n📝 Make sure:")
        print("   1. Google Drive is mounted (see Cell 2)")
        print("   2. Video file exists in your Google Drive")
        print("   3. Path is correct")
    else:
        print("\n📝 Please update the video_path variable above")
        print("   Example: video_path = '/path/to/your/video.mp4'")

In [ ]:
# Verify GPU before running inference
if not torch.cuda.is_available():
    print("⚠️ WARNING: No GPU detected!")
    print("   The inference will be very slow on CPU.")
    print("   Please go to Runtime > Change runtime type > Select GPU")
    proceed = input("Do you want to proceed anyway? (yes/no): ").lower() == 'yes'
    if not proceed:
        raise RuntimeError("GPU required for reasonable performance")
else:
    print(f"✓ GPU Ready: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()  # Clear any cached memory

# Set parameters for batch processing
model_path = "src/human3r.pth"
output_dir = "output"
json_output = "output/poses.json"  # JSON output path
size = 512
subsample = 1  # Use every frame (set higher to skip frames)
vis_threshold = 2.0
downsample_factor = 1
reset_interval = 100
batch_duration = 120.0  # Process in 120-second batches

print(f"📹 Processing video in {batch_duration}-second batches")
print("   This prevents GPU memory issues with long videos")
print("   Output: JSON only (video generation disabled to save memory)\n")

# Run inference with batch processing and JSON export
!python demo.py \
    --model_path {model_path} \
    --size {size} \
    --seq_path {video_path} \
    --output_dir {output_dir} \
    --json_output {json_output} \
    --subsample {subsample} \
    --batch_processing \
    --batch_duration {batch_duration} \
    --use_ttt3r \
    --vis_threshold {vis_threshold} \
    --downsample_factor {downsample_factor} \
    --reset_interval {reset_interval} \
    --device cuda

print("\n✅ Processing complete! JSON data exported to:", output_dir)

In [ ]:
# Download results
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

# Copy JSON to a convenient location or download
poses_json = os.path.join(output_dir, "poses.json")

if os.path.exists(poses_json):
    if IN_COLAB:
        # Google Colab: Auto-download
        from google.colab import files
        files.download(poses_json)
        print("📥 poses.json download started in browser")
    else:
        # Local machine: Show path
        json_path = os.path.abspath(poses_json)
        print(f"💾 JSON saved to: {json_path}")
        print(f"   File size: {os.path.getsize(poses_json) / (1024 * 1024):.2f} MB")
else:
    print("❌ poses.json not found")

## Advanced Options

You can modify the inference parameters in Cell 8:
- `subsample`: Process every N frames (higher = faster but less smooth)
- `vis_threshold`: Visualization confidence threshold (1-inf, higher = cleaner)
- `downsample_factor`: Point cloud density (higher = faster rendering)
- `reset_interval`: Reset tracking every N frames
- `size`: Input image size (224 or 512)

For more details, see the [GitHub repository](https://github.com/fanegg/Human3R).

---

## Citation

If you find this work useful, please cite:

```bibtex
@article{chen2025human3r,
    title={Human3R: Everyone Everywhere All at Once},
    author={Chen, Yue and Chen, Xingyu and Xue, Yuxuan and Chen, Anpei and Xiu, Yuliang and Gerard, Pons-Moll},
    journal={arXiv preprint arXiv:2510.06219},
    year={2025}
}
```

## 8. View Results

The processing is now complete! The results have been saved to the output directory:

**Exported Files:**
- `poses.json`: Structured JSON with camera parameters and human 3D poses in **camera space**

**Memory Optimization:**
- **No video generation**: Saves GPU memory and processing time
- **No frame rendering**: Saves GPU memory and disk space
- **JSON only**: Lightweight output (~5-15 MB per 1000 frames)

**What you get:**
- **Camera parameters** (focal length, principal point, rotation, translation) for each frame
- **SMPLX parameters** (shape, pose, translation) for each detected person in camera space
- **3D joint positions** (body, face, hands) for each person in **camera space**
- Person tracking IDs maintained across frames
- All data ready for world coordinate transformation in your external program

## Understanding the JSON Output Format (Camera Space)

The exported `poses.json` contains everything you need for 3D reconstruction with **full SMPLX parameters** and **camera-space joint positions**:

### Metadata
- Video information (path, fps, frames)
- **Coordinate system: CAMERA SPACE** (meters)
- Format version: **2.0** (camera-space coordinates)
- Processing parameters (subsample factor, batch info)
- Joint topology information (22 body + 3 face + 30 hand joints)

### Per-Frame Data
For each frame, you get:
- **Camera parameters**: 
  - `focal_length`: Focal length in pixels (may vary per frame for zoom)
  - `principal_point`: Image center [cx, cy]
  - `rotation_matrix`: 3x3 rotation matrix (camera to world)
  - `translation`: 3D camera position in world space
  - **Transform to world**: `p_world = R @ p_camera + t`
- **Human poses** (for each detected person):
  - `person_id`: Unique identifier (consistent across frames)
  - `camera_joints`: **3D positions in CAMERA SPACE**
    - `body`: 22x3 array of body joint positions
    - `face`: 3x3 array of face joint positions
    - `left_hand`: 15x3 array of left hand joint positions
    - `right_hand`: 15x3 array of right hand joint positions
  - `smplx_parameters`: 
    - **Shape**: 10D or 11D beta parameters
    - **Root pose**: 3D global orientation
    - **Body pose**: 21x3 rotation vectors (body joints)
    - **Translation**: 3D position in camera space
    - **Expression**: 10D facial expression (if available)
    - **Face**: jaw_pose, left_eye_pose, right_eye_pose (each 1x3)
    - **Hands**: left_hand_pose, right_hand_pose (each 15x3)

### Camera Space vs World Space
**All joint positions are in CAMERA SPACE** to give you full control:

- **Camera space**: Origin at camera, Z-axis pointing forward
- **World space**: Transform using camera parameters
- **Transformation**: `p_world = R @ p_camera + t`
  - `R`: Camera rotation matrix (3x3)
  - `t`: Camera translation (3D position in world)
  - `p_camera`: Joint position from JSON (3D)
  - `p_world`: Computed world position (3D)

**Why camera space?**
- Handles variable focal length (zoom) correctly
- Separates camera motion from human motion
- Gives you control over world coordinate frame
- Simpler and more accurate

### Quick Start: Transform to World Coordinates

```python
import json
import numpy as np

# Load poses
with open('output/poses.json', 'r') as f:
    data = json.load(f)

# Get frame 10 data
frame_10 = data['frames']['10']
camera = frame_10['camera']
human_0 = frame_10['humans'][0]

# Camera parameters
R = np.array(camera['rotation_matrix'])  # 3x3
t = np.array(camera['translation'])  # 3D
focal = camera['focal_length']  # scalar
pp = np.array(camera['principal_point'])  # 2D

# Get camera-space joints
joints_camera = np.array(human_0['camera_joints']['body'])  # 22x3

# Transform to world space
joints_world = (R @ joints_camera.T).T + t  # 22x3

print(f"Camera space pelvis: {joints_camera[0]}")
print(f"World space pelvis: {joints_world[0]}")
```

### Example: Visualize in Camera Space

```python
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Load poses
with open('output/poses.json', 'r') as f:
    data = json.load(f)

# Get joints for frame 10, person 0
frame_10 = data['frames']['10']
human_0 = frame_10['humans'][0]
camera_joints = human_0['camera_joints']

# Extract 3D positions (in camera space)
body_joints = np.array(camera_joints['body'])  # 22x3
face_joints = np.array(camera_joints['face'])  # 3x3
left_hand = np.array(camera_joints['left_hand'])  # 15x3
right_hand = np.array(camera_joints['right_hand'])  # 15x3

# Visualize in 3D (camera view)
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(body_joints[:, 0], body_joints[:, 1], body_joints[:, 2], 
           c='blue', s=50, label='Body')
ax.scatter(face_joints[:, 0], face_joints[:, 1], face_joints[:, 2], 
           c='red', s=50, label='Face')
ax.scatter(left_hand[:, 0], left_hand[:, 1], left_hand[:, 2], 
           c='green', s=30, label='Left Hand')
ax.scatter(right_hand[:, 0], right_hand[:, 1], right_hand[:, 2], 
           c='orange', s=30, label='Right Hand')

ax.set_xlabel('X (meters)')
ax.set_ylabel('Y (meters)')
ax.set_zlabel('Z (meters)')
ax.set_title('Camera Space View')
ax.legend()
plt.show()
```

### Example: Track in World Space

```python
import json
import numpy as np

with open('output/poses.json', 'r') as f:
    data = json.load(f)

# Track pelvis in world space across all frames
person_id = 0
trajectory_world = []

for frame_num, frame_data in data['frames'].items():
    camera = frame_data['camera']
    R = np.array(camera['rotation_matrix'])
    t = np.array(camera['translation'])
    
    for human in frame_data['humans']:
        if human['person_id'] == person_id:
            pelvis_camera = np.array(human['camera_joints']['body'][0])
            pelvis_world = R @ pelvis_camera + t
            
            trajectory_world.append({
                'frame': int(frame_num),
                'position': pelvis_world
            })

# Analyze movement
positions = np.array([t['position'] for t in trajectory_world])
distance = np.linalg.norm(positions[-1] - positions[0])
print(f"Person {person_id} traveled {distance:.2f} meters in world space")
```

### File Structure
```json
{
  "metadata": {
    "coordinate_system": "camera",
    "format_version": "2.0",
    "includes_camera_joints": true,
    ...
  },
  "frames": {
    "0": {
      "camera": {
        "focal_length": 512.5,
        "principal_point": [256, 256],
        "rotation_matrix": [[...], [...], [...]],
        "translation": [x, y, z]
      },
      "humans": [
        {
          "person_id": 0,
          "camera_joints": {
            "body": [[x,y,z], ...],  // 22 joints
            "face": [[x,y,z], ...],   // 3 joints
            "left_hand": [[x,y,z], ...],  // 15 joints
            "right_hand": [[x,y,z], ...]  // 15 joints
          },
          "smplx_parameters": { ... }
        }
      ]
    }
  }
}
```

### Advantages of Camera Space Export
1. **Accuracy**: No accumulated transformation errors
2. **Flexibility**: Choose your own world coordinate frame
3. **Zoom handling**: Focal length changes properly handled
4. **Debugging**: Easier to verify camera-space joint positions
5. **Simpler**: Clear separation of camera and human data